Copyright 2026 Snowflake Inc.
SPDX-License-Identifier: Apache-2.0

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

⚠️ **IMPORTANT**: This environment uses **Spark Connect** — a single Spark server runs in the background and all notebooks connect to it as lightweight clients (no per-notebook JVM). If you see `ConnectionRefusedError: [Errno 111] Connection refused` when initializing the Spark Session, the Spark Connect server may not be running. Check the Docker container logs with `docker logs jupyter-spark`. If the server failed to start, try restarting the container with `docker compose restart jupyter`.

In [1]:
# Initialize Spark session for data setup

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("Setup") \
    .getOrCreate()

/opt/conda/lib/python3.13/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at spark/connect/base.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/opt/conda/lib/python3.13/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at spark/connect/commands.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/opt/conda/lib/python3.13/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at spark/connect/common.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/opt/conda/lib/p

## Create Namespace

In [2]:
# Create the taxi namespace in Polaris catalog

spark.sql("CREATE NAMESPACE IF NOT EXISTS polaris.taxi")

DataFrame[]

## Download NYC Taxi Data

Download NYC Yellow Taxi trip data for June through October 2023 and upload to RustFS.

In [3]:
# Download NYC taxi data and upload to RustFS

import urllib.request
import os
import boto3
from botocore.client import Config

# Configure RustFS client
s3_client = boto3.client(
    's3',
    endpoint_url='http://rustfs:9000',
    aws_access_key_id='admin',
    aws_secret_access_key='password',
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

# Download and upload NYC taxi data
base_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-{:02d}.parquet"
bucket_name = "warehouse"
rustfs_prefix = "raw"
temp_dir = "/tmp/nyc_taxi_download"

os.makedirs(temp_dir, exist_ok=True)

months = range(6, 11)
uploaded_files = []

for month in months:
    url = base_url.format(month)
    filename = f"yellow_tripdata_2023-{month:02d}.parquet"
    local_path = os.path.join(temp_dir, filename)
    s3_key = f"{rustfs_prefix}/{filename}"
    
    print(f"Downloading {filename}...")
    urllib.request.urlretrieve(url, local_path)
    
    print(f"Uploading to RustFS: {s3_key}")
    s3_client.upload_file(local_path, bucket_name, s3_key)
    uploaded_files.append(s3_key)
    
    os.remove(local_path)

print(f"\nUploaded {len(uploaded_files)} files to s3://{bucket_name}/{rustfs_prefix}/")

Uploading to RustFS: raw/yellow_tripdata_2023-06.parquet
Uploading to RustFS: raw/yellow_tripdata_2023-07.parquet
Uploading to RustFS: raw/yellow_tripdata_2023-08.parquet
Uploading to RustFS: raw/yellow_tripdata_2023-09.parquet
Uploading to RustFS: raw/yellow_tripdata_2023-10.parquet

Uploaded 5 files to s3://warehouse/raw/


## Copy Data for Snapshot Example

Copy the raw data to a dedicated location for the snapshot example. This ensures data files are in the correct location when the snapshot table is created.

In [4]:
# Copy raw data to taxi/trips_snapshot/data/ for snapshot example

source_prefix = "raw/"
dest_prefix = "taxi/trips_snapshot/data/"

# List all files in the raw directory
response = s3_client.list_objects_v2(Bucket=bucket_name, Prefix=source_prefix)

if 'Contents' in response:
    for obj in response['Contents']:
        source_key = obj['Key']
        # Skip if it's just the directory marker
        if source_key.endswith('/'):
            continue
        
        # Get just the filename
        filename = os.path.basename(source_key)
        dest_key = f"{dest_prefix}{filename}"
        
        print(f"Copying {source_key} -> {dest_key}")
        s3_client.copy_object(
            Bucket=bucket_name,
            CopySource={'Bucket': bucket_name, 'Key': source_key},
            Key=dest_key
        )
    
    print(f"\nCopied data files to s3://{bucket_name}/{dest_prefix}")
else:
    print("No files found in raw directory")

Copying raw/yellow_tripdata_2023-06.parquet -> taxi/trips_snapshot/data/yellow_tripdata_2023-06.parquet
Copying raw/yellow_tripdata_2023-07.parquet -> taxi/trips_snapshot/data/yellow_tripdata_2023-07.parquet
Copying raw/yellow_tripdata_2023-08.parquet -> taxi/trips_snapshot/data/yellow_tripdata_2023-08.parquet
Copying raw/yellow_tripdata_2023-09.parquet -> taxi/trips_snapshot/data/yellow_tripdata_2023-09.parquet
Copying raw/yellow_tripdata_2023-10.parquet -> taxi/trips_snapshot/data/yellow_tripdata_2023-10.parquet

Copied data files to s3://warehouse/taxi/trips_snapshot/data/


## Verify Data

Verify both data locations are accessible.

In [5]:
# Verify raw data location

raw_df = spark.read.parquet("s3a://warehouse/raw/")
print(f"Raw data: {raw_df.count():,} rows")

# Verify taxi/trips_snapshot data location
trips_snapshot_df = spark.read.parquet("s3a://warehouse/taxi/trips_snapshot/data/")
print(f"Trips snapshot data: {trips_snapshot_df.count():,} rows")

Raw data: 15,407,558 rows
Trips snapshot data: 15,407,558 rows
